# Проверка фикса озера: май 2026 (IRF + trx)

Не берём старый `final_df` / checkpoint — только живые `scd1_trx` + `scd1_trx_int` после invalidate.

| Было (до фикса) | Ждём |
|---|---|
| тотал trx lake/excel ≈ 0.97 | ≈ 1.00 |
| exact `trx_cnt`/`trx_sum` ~55% | как июнь, ~99% |
| IRF coverage дыра при живых trx | как апрель/июнь |
| июнь | без регресса |

Периметр = секция 05: SA / S01 / не `R` / есть терминал / эквайер RSHB / SA-договор.

`run_august_irf_impute` не включаем.

После Run All ячейка VERDICT пишет `qc_may_lake_fix/may_lake_fix_verdict.txt` (`GREEN` / `RED`). `final_script_2` при `GREEN` сам пересчитает только май (не старый checkpoint).


In [ ]:
import re
from calendar import monthrange
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd()
OUT_DIR = DATA_DIR / 'qc_may_lake_fix'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_MONTH = '2026-05'
CONTROL_MONTH = '2026-06'
PEER_MONTH = '2026-04'
MONTHS = [PEER_MONTH, TARGET_MONTH, CONTROL_MONTH]

excel_by_month = {
    '2026-04': DATA_DIR / '04_Апрель_2026.xlsx',
    '2026-05': DATA_DIR / '05_Май_2026.xlsx',
    '2026-06': DATA_DIR / '06_Июнь_2026.xlsx',
}
excel_header_by_month = {'2026-01': 1, '2026-02': 1}

run_invalidate = True
MEM_LIMIT = '8g'
NOTEBOOK_REV = '2026-09-25-may-lake-fix-v1'

print('rev', NOTEBOOK_REV)
print('months', MONTHS)
print('OUT_DIR', OUT_DIR)


## 0) Impala + invalidate / refresh


In [ ]:
if 'imp' in globals() and imp is not None:
    print('Reuse Impala')
else:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'},
    )
    imp._init_connection()
    print('Impala connected')

if run_invalidate:
    with imp:
        for t in (
            'ods_alpha.scd1_trx',
            'ods_alpha.scd1_trx_int',
            'ods_alpha.scd1_trx_acq',
            'ods_alpha.scd1_base24_fiids',
            'ods_alpha.scd1_agreements',
            'ods_alpha.scd1_companies',
        ):
            try:
                imp.execute('invalidate metadata ' + t)
                imp.execute('refresh ' + t)
                print('[ok]', t)
            except Exception as exc:
                print('[fail]', t, type(exc).__name__, str(exc)[:200])


## 1) Probe май vs апрель / июнь (без Excel)

`trx_cnt` = distinct `n_trx`, `trx_sum` = сумма `n_amt_src`.  
`int_coverage_pct` = доля trx со строкой в `scd1_trx_int`.


In [ ]:
def month_bounds(ym):
    y, m = map(int, ym.split('-'))
    start = f'{ym}-01'
    last = monthrange(y, m)[1]
    end_excl = f'{y}-{m+1:02d}-01' if m < 12 else f'{y+1}-01-01'
    return start, end_excl, f'{ym}-{last:02d}'


def fetch(sql, label):
    print(f'[{pd.Timestamp.now().strftime("%H:%M:%S")}] {label}', flush=True)
    with imp:
        try:
            imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
        except Exception:
            pass
        out = imp.fetch(sql)
    return pd.DataFrame() if out is None else out


def sql_month_probe(ym):
    start, end_excl, _ = month_bounds(ym)
    return f'''
    WITH fiid_rshb AS (
      SELECT DISTINCT CAST(fa.c_fiid AS STRING) AS c_fiid
      FROM ods_alpha.scd1_base24_fiids fa
      WHERE COALESCE(CAST(fa.c_fiid_grp AS STRING), 'UNKNOWN') = 'RSHB'
    ),
    sa_agr AS (
      SELECT DISTINCT CAST(a.n_agr AS STRING) AS n_agr
      FROM ods_alpha.scd1_agreements a
      WHERE UPPER(TRIM(CAST(a.acq_class AS STRING))) = 'SA'
        AND a.abs_agr_id IS NOT NULL
        AND COALESCE(a.ods_deleted_flg, '0') <> '1'
    ),
    trx_base AS (
      SELECT CAST(t.n_trx AS STRING) AS n_trx, MAX(CAST(t.n_amt_src AS DOUBLE)) AS n_amt_src
      FROM ods_alpha.scd1_trx t
      JOIN fiid_rshb fr ON fr.c_fiid = CAST(t.c_fiid_acq AS STRING)
      WHERE CAST(t.d_trx_orig AS TIMESTAMP) >= CAST('{start}' AS TIMESTAMP)
        AND CAST(t.d_trx_orig AS TIMESTAMP) < CAST('{end_excl}' AS TIMESTAMP)
        AND t.c_nter IS NOT NULL
        AND COALESCE(t.ods_deleted_flg, '0') <> '1'
        AND t.c_trx_class = 'SA'
        AND t.c_trx_type = 'S01'
        AND COALESCE(t.cf_trx_stat, '') <> 'R'
      GROUP BY CAST(t.n_trx AS STRING)
    ),
    ta AS (
      SELECT CAST(a.n_trx AS STRING) AS n_trx
      FROM ods_alpha.scd1_trx_acq a
      JOIN trx_base tb ON tb.n_trx = CAST(a.n_trx AS STRING)
      JOIN sa_agr ss ON ss.n_agr = CAST(a.n_agr AS STRING)
      WHERE COALESCE(a.ods_deleted_flg, '0') <> '1'
      GROUP BY CAST(a.n_trx AS STRING)
    )
    SELECT
      '{ym}' AS report_month,
      COUNT(*) AS trx_cnt,
      SUM(tb.n_amt_src) AS trx_sum,
      SUM(CASE WHEN ti.n_trx IS NOT NULL THEN 1 ELSE 0 END) AS with_int_row,
      SUM(COALESCE(CAST(ti.n_amt_fee AS DOUBLE), 0.0)) AS sum_n_amt_fee,
      SUM(ABS(COALESCE(CAST(ti.n_amt_fee AS DOUBLE), 0.0))) AS sum_n_amt_fee_abs
    FROM trx_base tb
    JOIN ta ON ta.n_trx = tb.n_trx
    LEFT JOIN ods_alpha.scd1_trx_int ti
      ON CAST(ti.n_trx AS STRING) = tb.n_trx
     AND COALESCE(ti.ods_deleted_flg, '0') <> '1'
    '''


probe_parts = []
for ym in MONTHS:
    part = fetch(sql_month_probe(ym), f'probe {ym}')
    probe_parts.append(part)
    print(f'  {ym}: rows_returned={len(part)}')

probe = pd.concat(probe_parts, ignore_index=True)
for c in ('trx_cnt', 'trx_sum', 'with_int_row', 'sum_n_amt_fee', 'sum_n_amt_fee_abs'):
    probe[c] = pd.to_numeric(probe[c], errors='coerce')
probe['int_coverage_pct'] = np.where(probe['trx_cnt'] > 0, 100.0 * probe['with_int_row'] / probe['trx_cnt'], np.nan)
print('=== Probe ODS (секция 05) ===')
display(probe)

may = probe.loc[probe['report_month'] == TARGET_MONTH]
jun = probe.loc[probe['report_month'] == CONTROL_MONTH]
apr = probe.loc[probe['report_month'] == PEER_MONTH]
if len(may) and len(jun):
    print('coverage май', round(float(may['int_coverage_pct'].iloc[0]), 2),
          '| июнь', round(float(jun['int_coverage_pct'].iloc[0]), 2),
          '| апрель', round(float(apr['int_coverage_pct'].iloc[0]), 2) if len(apr) else None)


## 2) Excel: тоталы trx / IRF (inn+agr: max cnt, sum sum)


In [ ]:
def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = re.sub(r'\D+', '', re.sub(r'\.0$', '', str(v).strip()))
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str).str.replace('\xa0', '', regex=False).str.replace(' ', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce',
    )


def pick_col(columns, needles):
    cols = list(columns)
    norm = lambda x: re.sub(r'\s+', ' ', str(x).replace('\xa0', ' ').replace('\n', ' ').strip().lower())
    nmap = {norm(c): c for c in cols}
    for n in needles:
        if n in cols:
            return n
        nn = norm(n)
        if nn in nmap:
            return nmap[nn]
        for k, v in nmap.items():
            if nn in k or k in nn:
                return v
    return None


INN_CAND = ['ИНН', 'inn', 'c_inn']
AGR_CAND = ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id']
CNT_CAND = ['Количество операций', 'Количеств операций', 'trx_cnt']
SUM_CAND = ['Сумма операций', 'Сумма опреаций', 'trx_sum']
OPS_CAND = ['Комиссия эквайринга', 'Комиссия (% с операций)', 'Комиссия % с операций']
IRF_CAND = ['Комиссия МПС (IRF, ₽)', 'Комиссия МПС (IRF, р)', 'Комиссия МПС (IRF)', 'IRF']


def load_excel_month(ym):
    path = excel_by_month[ym]
    header = int(excel_header_by_month.get(ym, 0))
    rec = {'report_month': ym, 'excel_exists': path.exists(), 'excel_path': path.name}
    if not path.exists():
        rec['excel_error'] = 'missing file'
        return rec, None
    ex = pd.read_excel(path, header=header)
    c_inn = pick_col(ex.columns, INN_CAND)
    c_agr = pick_col(ex.columns, AGR_CAND)
    c_cnt = pick_col(ex.columns, CNT_CAND)
    c_sum = pick_col(ex.columns, SUM_CAND)
    c_ops = pick_col(ex.columns, OPS_CAND)
    c_irf = pick_col(ex.columns, IRF_CAND)
    rec.update(inn_col=c_inn, agr_col=c_agr, cnt_col=c_cnt, sum_col=c_sum, irf_col=c_irf)
    if None in (c_inn, c_agr, c_cnt, c_sum):
        rec['excel_error'] = f'missing cols inn={c_inn} agr={c_agr} cnt={c_cnt} sum={c_sum}'
        return rec, None
    tmp = pd.DataFrame({
        'inn_key': ex[c_inn].map(normalize_inn_q1),
        'agr_id_key': ex[c_agr].map(normalize_agr_q1),
        'trx_cnt_excel': pd.to_numeric(ex[c_cnt], errors='coerce'),
        'trx_sum_excel': to_num_series(ex[c_sum]),
        'commission_from_ops_excel': to_num_series(ex[c_ops]) if c_ops else np.nan,
        'irf_excel': to_num_series(ex[c_irf]) if c_irf else np.nan,
    })
    g = (
        tmp.dropna(subset=['inn_key', 'agr_id_key'])
        .groupby(['inn_key', 'agr_id_key'], as_index=False)
        .agg(
            trx_cnt_excel=('trx_cnt_excel', 'max'),
            trx_sum_excel=('trx_sum_excel', 'sum'),
            commission_from_ops_excel=('commission_from_ops_excel', 'sum'),
            irf_excel=('irf_excel', 'sum'),
        )
    )
    rec['excel_error'] = ''
    rec['excel_keys'] = len(g)
    rec['trx_cnt_excel'] = float(g['trx_cnt_excel'].fillna(0).sum())
    rec['trx_sum_excel'] = float(g['trx_sum_excel'].fillna(0).sum())
    rec['ops_excel'] = float(g['commission_from_ops_excel'].fillna(0).sum())
    rec['irf_excel'] = float(g['irf_excel'].fillna(0).sum())
    rec['irf_excel_abs'] = float(g['irf_excel'].abs().fillna(0).sum())
    print(f'Excel {ym}: keys={rec["excel_keys"]:,} cnt={rec["trx_cnt_excel"]:,.0f} sum={rec["trx_sum_excel"]:,.2f} IRF={rec["irf_excel"]:,.2f} col={c_irf}')
    return rec, g


excel_totals = []
excel_agg = {}
for ym in MONTHS:
    rec, g = load_excel_month(ym)
    excel_totals.append(rec)
    if g is not None:
        excel_agg[ym] = g
excel_tot = pd.DataFrame(excel_totals)
display(excel_tot)


## 3) ODS по inn+agr (май и июнь) — exact match vs Excel

Не checkpoint. Один запрос на месяц, зерно как в QC: `inn` + `abs_agr_id`.


In [ ]:
def sql_agr_month(ym):
    start, end_excl, _ = month_bounds(ym)
    return f'''
    WITH fiid_rshb AS (
      SELECT DISTINCT CAST(fa.c_fiid AS STRING) AS c_fiid
      FROM ods_alpha.scd1_base24_fiids fa
      WHERE COALESCE(CAST(fa.c_fiid_grp AS STRING), 'UNKNOWN') = 'RSHB'
    ),
    sa_agr AS (
      SELECT
        CAST(a.n_agr AS STRING) AS n_agr,
        CAST(a.abs_agr_id AS STRING) AS agr_id,
        CAST(a.n_cmp_client AS STRING) AS n_cmp_client
      FROM ods_alpha.scd1_agreements a
      WHERE UPPER(TRIM(CAST(a.acq_class AS STRING))) = 'SA'
        AND a.abs_agr_id IS NOT NULL
        AND COALESCE(a.ods_deleted_flg, '0') <> '1'
    ),
    trx_base AS (
      SELECT CAST(t.n_trx AS STRING) AS n_trx, MAX(CAST(t.n_amt_src AS DOUBLE)) AS n_amt_src
      FROM ods_alpha.scd1_trx t
      JOIN fiid_rshb fr ON fr.c_fiid = CAST(t.c_fiid_acq AS STRING)
      WHERE CAST(t.d_trx_orig AS TIMESTAMP) >= CAST('{start}' AS TIMESTAMP)
        AND CAST(t.d_trx_orig AS TIMESTAMP) < CAST('{end_excl}' AS TIMESTAMP)
        AND t.c_nter IS NOT NULL
        AND COALESCE(t.ods_deleted_flg, '0') <> '1'
        AND t.c_trx_class = 'SA'
        AND t.c_trx_type = 'S01'
        AND COALESCE(t.cf_trx_stat, '') <> 'R'
      GROUP BY CAST(t.n_trx AS STRING)
    ),
    ta AS (
      SELECT
        CAST(a.n_trx AS STRING) AS n_trx,
        CAST(a.n_agr AS STRING) AS n_agr,
        MAX(COALESCE(CAST(a.n_amt_tax AS DOUBLE), 0.0)) AS n_amt_tax
      FROM ods_alpha.scd1_trx_acq a
      JOIN trx_base tb ON tb.n_trx = CAST(a.n_trx AS STRING)
      JOIN sa_agr ss ON ss.n_agr = CAST(a.n_agr AS STRING)
      WHERE COALESCE(a.ods_deleted_flg, '0') <> '1'
      GROUP BY CAST(a.n_trx AS STRING), CAST(a.n_agr AS STRING)
    )
    SELECT
      '{ym}' AS report_month,
      regexp_replace(trim(cast(c.c_inn AS string)), '[^0-9]', '') AS inn,
      ss.agr_id AS agr_id,
      COUNT(DISTINCT tb.n_trx) AS trx_cnt,
      SUM(tb.n_amt_src) AS trx_sum,
      SUM(ta.n_amt_tax) AS commission_from_ops,
      SUM(COALESCE(CAST(ti.n_amt_fee AS DOUBLE), 0.0)) AS irf_sum,
      SUM(ABS(COALESCE(CAST(ti.n_amt_fee AS DOUBLE), 0.0))) AS irf_abs
    FROM trx_base tb
    JOIN ta ON ta.n_trx = tb.n_trx
    JOIN sa_agr ss ON ss.n_agr = ta.n_agr
    LEFT JOIN ods_alpha.scd1_companies c
      ON CAST(c.n_cmp AS STRING) = ss.n_cmp_client
     AND COALESCE(c.ods_deleted_flg, '0') <> '1'
    LEFT JOIN ods_alpha.scd1_trx_int ti
      ON CAST(ti.n_trx AS STRING) = tb.n_trx
     AND COALESCE(ti.ods_deleted_flg, '0') <> '1'
    GROUP BY 1, 2, 3
    '''


def build_lake_keys(df):
    out = df.copy()
    out['inn_key'] = out['inn'].map(normalize_inn_q1)
    out['agr_id_key'] = out['agr_id'].map(normalize_agr_q1)
    for c in ('trx_cnt', 'trx_sum', 'commission_from_ops', 'irf_sum', 'irf_abs'):
        out[c] = pd.to_numeric(out[c], errors='coerce')
    return (
        out.dropna(subset=['inn_key', 'agr_id_key'])
        .groupby(['inn_key', 'agr_id_key'], as_index=False)
        .agg(
            trx_cnt_lake=('trx_cnt', 'sum'),
            trx_sum_lake=('trx_sum', 'sum'),
            ops_lake=('commission_from_ops', 'sum'),
            irf_lake=('irf_sum', 'sum'),
            irf_abs_lake=('irf_abs', 'sum'),
        )
    )


def exact_pct(a, b, atol=0.01):
    a = pd.to_numeric(a, errors='coerce')
    b = pd.to_numeric(b, errors='coerce')
    both_na = a.isna() & b.isna()
    ok = both_na | ((a - b).abs() <= atol) | (a == b)
    return 100.0 * float(ok.mean()) if len(ok) else np.nan


lake_agg = {}
for ym in (TARGET_MONTH, CONTROL_MONTH):
    raw = fetch(sql_agr_month(ym), f'agr {ym}')
    print(f'  {ym}: ods agr rows={len(raw):,}')
    lake_agg[ym] = build_lake_keys(raw)
    print(f'  {ym}: keys={len(lake_agg[ym]):,}')
    display(lake_agg[ym].head(3))


## 4) Сводка Excel vs живой ODS


In [ ]:
rows = []
cmp_store = {}
for ym in (TARGET_MONTH, CONTROL_MONTH):
    lk = lake_agg[ym]
    ex = excel_agg.get(ym)
    rec = {'report_month': ym}
    rec['trx_cnt_lake'] = float(lk['trx_cnt_lake'].sum())
    rec['trx_sum_lake'] = float(lk['trx_sum_lake'].sum())
    rec['irf_abs_lake'] = float(lk['irf_abs_lake'].sum())
    rec['keys_lake'] = len(lk)
    if ex is None:
        rec['note'] = 'no excel'
        rows.append(rec)
        continue
    m = lk.merge(ex, on=['inn_key', 'agr_id_key'], how='outer', indicator=True)
    both = m[m['_merge'] == 'both']
    rec['keys_excel'] = len(ex)
    rec['keys_both'] = len(both)
    rec['trx_cnt_excel'] = float(ex['trx_cnt_excel'].fillna(0).sum())
    rec['trx_sum_excel'] = float(ex['trx_sum_excel'].fillna(0).sum())
    rec['irf_excel'] = float(ex['irf_excel'].fillna(0).sum())
    rec['irf_excel_abs'] = float(ex['irf_excel'].abs().fillna(0).sum())
    rec['ratio_cnt'] = rec['trx_cnt_lake'] / rec['trx_cnt_excel'] if rec['trx_cnt_excel'] else np.nan
    rec['ratio_sum'] = rec['trx_sum_lake'] / rec['trx_sum_excel'] if rec['trx_sum_excel'] else np.nan
    rec['ratio_irf_abs'] = rec['irf_abs_lake'] / rec['irf_excel_abs'] if rec['irf_excel_abs'] else np.nan
    rec['exact_trx_cnt_pct'] = exact_pct(both['trx_cnt_lake'], both['trx_cnt_excel'], atol=0)
    rec['exact_trx_sum_pct'] = exact_pct(both['trx_sum_lake'], both['trx_sum_excel'], atol=0.01)
    rec['exact_ops_pct'] = exact_pct(both['ops_lake'], both['commission_from_ops_excel'], atol=0.01)
    if len(both):
        ratio = both['trx_sum_lake'] / both['trx_sum_excel'].replace(0, np.nan)
        rec['median_sum_ratio_both'] = float(ratio.median())
        rec['share_ratio_0_9_1_1'] = 100.0 * float(((ratio >= 0.9) & (ratio <= 1.1)).mean())
    cmp_store[ym] = m
    rows.append(rec)

summary = pd.DataFrame(rows)
print('=== Excel vs ODS (не final_df) ===')
display(summary)

print('=== Probe coverage ===')
display(probe[['report_month', 'trx_cnt', 'trx_sum', 'int_coverage_pct', 'sum_n_amt_fee_abs']])


## 5) VERDICT


In [ ]:
def _row(ym):
    s = summary.loc[summary['report_month'] == ym]
    return s.iloc[0] if len(s) else None


may_s = _row(TARGET_MONTH)
jun_s = _row(CONTROL_MONTH)
may_p = probe.loc[probe['report_month'] == TARGET_MONTH]
jun_p = probe.loc[probe['report_month'] == CONTROL_MONTH]
apr_p = probe.loc[probe['report_month'] == PEER_MONTH]

lines = []
ok_all = True

def check(name, cond, detail):
    global ok_all
    flag = 'OK' if cond else 'FAIL'
    if not cond:
        ok_all = False
    lines.append(f'{flag}  {name}: {detail}')


if may_s is not None:
    check(
        'май trx_cnt ratio',
        pd.notna(may_s['ratio_cnt']) and abs(may_s['ratio_cnt'] - 1.0) <= 0.02,
        f"{may_s['ratio_cnt']:.4f} (ждали ≈1.00, было ~0.97)",
    )
    check(
        'май trx_sum ratio',
        pd.notna(may_s['ratio_sum']) and abs(may_s['ratio_sum'] - 1.0) <= 0.02,
        f"{may_s['ratio_sum']:.4f}",
    )
    check(
        'май exact trx_cnt %',
        pd.notna(may_s['exact_trx_cnt_pct']) and may_s['exact_trx_cnt_pct'] >= 90,
        f"{may_s['exact_trx_cnt_pct']:.1f}% (было ~55%, июнь-ориентир ≥90)",
    )
    check(
        'май exact trx_sum %',
        pd.notna(may_s['exact_trx_sum_pct']) and may_s['exact_trx_sum_pct'] >= 90,
        f"{may_s['exact_trx_sum_pct']:.1f}%",
    )

if len(may_p) and len(jun_p):
    cov_m = float(may_p['int_coverage_pct'].iloc[0])
    cov_j = float(jun_p['int_coverage_pct'].iloc[0])
    cov_a = float(apr_p['int_coverage_pct'].iloc[0]) if len(apr_p) else cov_j
    check(
        'май IRF coverage vs соседи',
        cov_m >= 50 and abs(cov_m - cov_j) <= 15,
        f"май {cov_m:.1f}% | июнь {cov_j:.1f}% | апрель {cov_a:.1f}%",
    )

if may_s is not None and pd.notna(may_s.get('ratio_irf_abs')):
    check(
        'май |IRF| vs Excel',
        abs(may_s['ratio_irf_abs'] - 1.0) <= 0.15,
        f"{may_s['ratio_irf_abs']:.4f} (допуск 15% на знак/агрегат)",
    )

if jun_s is not None:
    check(
        'июнь без регресса (sum ratio)',
        pd.notna(jun_s['ratio_sum']) and abs(jun_s['ratio_sum'] - 1.0) <= 0.03,
        f"{jun_s['ratio_sum']:.4f}",
    )
    check(
        'июнь exact trx_sum %',
        pd.notna(jun_s['exact_trx_sum_pct']) and jun_s['exact_trx_sum_pct'] >= 90,
        f"{jun_s['exact_trx_sum_pct']:.1f}%",
    )

print('=== VERDICT ===')
for ln in lines:
    print(ln)
print()
print('ИТОГО:', 'ФИКС ПОДТВЕРЖДЁН' if ok_all else 'ФИКС НЕ ПОДТВЕРЖДЁН — смотри FAIL')

verdict_path = OUT_DIR / 'may_lake_fix_verdict.txt'
verdict_path.write_text(
    ('GREEN' if ok_all else 'RED') + '\n' + '\n'.join(lines) + '\n',
    encoding='utf-8',
)
print('verdict file:', verdict_path, '=', 'GREEN' if ok_all else 'RED')
if ok_all:
    print('NEXT: final_script_2 подхватит GREEN и пересчитает только 2026-05 (не старый checkpoint).')
    print('Старый final_df_2026_05.* будет стёрт. Остальные месяцы — с checkpoint.')
else:
    print('Витрину дашборда / final_df Jan–Aug не пересобирать, пока есть FAIL.')

summary.to_csv(OUT_DIR / 'may_lake_fix_summary.csv', index=False, encoding='utf-8-sig')
probe.to_csv(OUT_DIR / 'may_lake_fix_probe.csv', index=False, encoding='utf-8-sig')
print('saved', OUT_DIR)
